# **(ADD THE NOTEBOOK NAME HERE)**

## Objectives

* Load the cleaned dataset used in the EDA notebook
* Test H3 by comparing model performance using Risk_Score and Failed_Transaction_Count_7d individually against using them combined
* Train and compare Logistic Regression and Random Forest classification models to predict fraud
* Evaluate model performance using precision, recall and AUC, and identify the most important predictive features

## Inputs

* Cleaned dataset: Dataset/CleanData/cleaned_fraud_dataset.csv
* Python libraries: Pandas, Numpy, Scikit-learn

## Outputs

* A trained classification model and its evaluation results, to be summarised in the README
* A feature importance chart showing which variables the model relied on most

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here.


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/Fraud-Analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/Fraud-Analysis'

#  H3 — Does combining Risk Score and Failed Transaction Count 7d predict fraud more accurately than either feature alone? 
### Comparing Logistic Regression against Random Forest as a broader classification model.



### Import libraries

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

### Load Clean Dataset

In [5]:
# Load the cleaned dataset in
df = pd.read_csv('Dataset/CleanData/cleaned_fraud_dataset.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset loaded: 50000 rows, 20 columns


,Transaction_ID,Transaction_Amount,Transaction_Type,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend,Fraud_Label,hour,day_of_week
0,TXN_33553,39.79,POS,93213.17,Laptop,Sydney,Travel,0,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0,0,19,Monday
1,TXN_9427,1.19,Bank Transfer,75725.25,Mobile,New York,Clothing,0,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0,1,4,Wednesday
2,TXN_199,28.96,Online,1588.96,Tablet,Mumbai,Restaurants,0,14,50.01,4,Visa,226,1909.29,Biometric,0.8400,0,1,15,Tuesday
3,TXN_12447,254.32,ATM Withdrawal,76807.20,Tablet,New York,Clothing,0,8,182.48,4,Visa,76,1311.86,OTP,0.7935,0,1,0,Thursday
4,TXN_39489,31.28,POS,92354.66,Mobile,Mumbai,Electronics,1,14,328.69,4,Mastercard,140,966.98,Password,0.3819,1,1,23,Saturday


# H3 - Combined risk signal

Combining risk score and failed transaction count predicts fraud more accurately than either alone.

H0: A model using Risk_Score and Failed_Transaction_Count_7d together predicts fraud no better than a model using only one of them.

H1: The combined model predicts fraud significantly better than either single-variable model.

## Prepare Data for H3

Splitting the data into training and test sets, so each model can be evaluated on transactions it hasn't seen before.

In [6]:
# Splitting the dataset into features and target variable
X = df[['Risk_Score', 'Failed_Transaction_Count_7d']]
y = df['Fraud_Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Single-Feature Models

I will train two separate models each using only one feature to establish a baseline before testing the combined model.

In [7]:
# Training a Logistic Regression model using only the Risk_Score feature
model_risk = LogisticRegression()
model_risk.fit(X_train[['Risk_Score']], y_train)
pred_risk = model_risk.predict(X_test[['Risk_Score']])
print("Risk_Score only")
print(classification_report(y_test, pred_risk))
print("AUC:", roc_auc_score(y_test, model_risk.predict_proba(X_test[['Risk_Score']])[:,1]))

Risk_Score only
              precision    recall  f1-score   support

           0       0.80      0.92      0.86      6787
           1       0.76      0.52      0.62      3213

    accuracy                           0.79     10000
   macro avg       0.78      0.72      0.74     10000
weighted avg       0.79      0.79      0.78     10000

AUC: 0.7397023180701321


In [8]:
# Training a Logistic Regression model using only the Failed_Transaction_Count_7d feature
model_failed = LogisticRegression()
model_failed.fit(X_train[['Failed_Transaction_Count_7d']], y_train)
pred_failed = model_failed.predict(X_test[['Failed_Transaction_Count_7d']])
print("Failed_Transaction_Count_7d only")
print(classification_report(y_test, pred_failed))
print("AUC:", roc_auc_score(y_test, model_failed.predict_proba(X_test[['Failed_Transaction_Count_7d']])[:,1]))

Failed_Transaction_Count_7d only
              precision    recall  f1-score   support

           0       0.85      1.00      0.92      6787
           1       1.00      0.62      0.76      3213

    accuracy                           0.88     10000
   macro avg       0.92      0.81      0.84     10000
weighted avg       0.90      0.88      0.87     10000

AUC: 0.8026059366987959


## Combined-Feature Model

In [9]:
# Training a Logistic Regression model using both features
model_combined = LogisticRegression()
model_combined.fit(X_train, y_train)
pred_combined = model_combined.predict(X_test)
print("Combined")
print(classification_report(y_test, pred_combined))
print("AUC:", roc_auc_score(y_test, model_combined.predict_proba(X_test)[:,1]))

Combined
              precision    recall  f1-score   support

           0       0.84      0.87      0.86      6787
           1       0.71      0.64      0.67      3213

    accuracy                           0.80     10000
   macro avg       0.77      0.76      0.76     10000
weighted avg       0.80      0.80      0.80     10000

AUC: 0.8901322721515303


## H3 Result

| Model | AUC | Accuracy | Precision (fraud) | Recall (fraud) |

Risk_Score only | 0.740 | 0.79 | 0.76 | 0.52 |

Failed_Transaction_Count_7d only | 0.803 | 0.88 | 1.00 | 0.62 |

Combined | 0.890 | 0.80 | 0.71 | 0.64 |

Result: Supported

Summary: The combined model achieved a noticeably higher AUC (0.890) than either single-feature model (Risk_Score alone: 0.740, Failed_Transaction_Count_7d alone: 0.803), confirming that using both features together captures more information about fraud risk than either does on its own. Interestingly, the Failed_Transaction_Count_7d-only model had a higher raw accuracy (0.88) than the combined model (0.80). This is because Failed_Transaction_Count_7d alone is an extremely strong predictor in this dataset (a count of 4 was fraud 100% of the time, per the H1 visualisation). AUC is a more reliable measure of overall predictive power here since it isn't affected by the classification threshold, and it clearly shows the combined model separates fraud from genuine transactions more effectively.

# Logistic Regression vs Random Forest

Now testing a wider set of features, including categorical columns to compare two full classification models rather than just the two H3 variables. Since some features are categorical (text), a pipeline is used to convert them to numbers before training.

In [10]:
categorical_features = ['Transaction_Type', 'Location', 'Device_Type', 'Card_Type', 'Authentication_Method']
numeric_features = ['Risk_Score', 'Failed_Transaction_Count_7d', 'Transaction_Amount']
features = categorical_features + numeric_features

X_full = df[features]
y_full = df['Fraud_Label']

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
    remainder='passthrough'
)

In [11]:
logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

logreg_pipeline.fit(X_train_full, y_train_full)
pred_logreg = logreg_pipeline.predict(X_test_full)
print("Logistic Regression")
print(classification_report(y_test_full, pred_logreg))
print("AUC:", roc_auc_score(y_test_full, logreg_pipeline.predict_proba(X_test_full)[:,1]))

Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.87      0.86      6787
           1       0.71      0.64      0.67      3213

    accuracy                           0.80     10000
   macro avg       0.77      0.76      0.76     10000
weighted avg       0.80      0.80      0.80     10000

AUC: 0.8897712810383226


In [12]:
# Training a Random Forest Classifier
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_pipeline.fit(X_train_full, y_train_full)
pred_rf = rf_pipeline.predict(X_test_full)
print("Random Forest")
print(classification_report(y_test_full, pred_rf))
print("AUC:", roc_auc_score(y_test_full, rf_pipeline.predict_proba(X_test_full)[:,1]))

Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6787
           1       1.00      1.00      1.00      3213

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

AUC: 1.0


### Feature Importance

Checking which features the Random Forest model relied on most to see whether this matches the findings from the EDA and hypothesis testing.

In [13]:
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = pd.Series(rf_pipeline.named_steps['classifier'].feature_importances_, index=feature_names)
importances.sort_values(ascending=False).head(10).plot(kind='barh')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

<Figure size 640x480 with 1 Axes>

# Model Comparison Summary

| Model | AUC | Accuracy | Precision (fraud) | Recall (fraud) |
|---|---|---|---|---|
| Logistic Regression | 0.890 | 0.80 | 0.71 | 0.64 |
| Random Forest | 1.00 | 1.00 | 1.00 | 1.00 |

Random Forest achieved a perfect AUC of 1.00 on the test set, correctly classifying every transaction. While this looks impressive, a perfect score is a signal worth investigating rather than simply reporting as a win. Looking back at the EDA findings, Failed_Transaction_Count_7d reaching 4 corresponded to fraud 100% of the time in this dataset, and Risk_Score above 0.85 was exclusively seen in fraud cases too meaning this dataset contains near-deterministic rules that a flexible model like Random Forest can learn almost exactly. This likely reflects how the dataset was generated (synthetically, with clean underlying rules) rather than how real-world fraud typically behaves, where signals are noisier and rarely this clean-cut. The feature importance results reinforce this: Failed_Transaction_Count_7d (0.547) and Risk_Score (0.419) account for over 96% of the model's decision-making, with every other feature (device, card type, location, authentication method) contributing almost nothing.

For this reason, I'd recommend Logistic Regression as the more honest generalisable model to report — its AUC of 0.89 is still strong, its coefficients are interpretable, and it doesn't risk overfitting to artefacts specific to this synthetic dataset. Random Forest's perfect score is presented here for transparency, but should be treated cautiously rather than as evidence the problem is "solved."

# Saving for export

I am going to save the feature importance dataset into my cleaned data so that I can access it Tableau

In [14]:
importances.sort_values(ascending=False).to_csv('Dataset/CleanData/feature_importance.csv')

Exporting a small summary table comparing Logistic Regression and Random Forest on AUC, precision and recall so this comparison can be visualised in Tableau.

In [15]:
model_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'AUC': [0.890, 1.00],
    'Precision': [0.71, 1.00],
    'Recall': [0.64, 1.00]
})

model_comparison.to_csv('Dataset/CleanData/model_comparison.csv', index=False)
model_comparison

,Model,AUC,Precision,Recall
0,Logistic Regression,0.89,0.71,0.64
1,Random Forest,1.00,1.00,1.00


---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [16]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)